# A medicine with two effects on readmission and death: separable effects with TMLE

## At a glance

A heart-failure clinic sends patients home on no water pill, a standard dose or a high dose, as
their doctors choose. In this simulated clinic the pill has two parts: one clears fluid, which
keeps patients out of hospital, and one lowers blood potassium, which can cause a fatal heart
rhythm. So part of any drop in readmissions could be patients who died before they could be
readmitted. How much of the drop would remain if a potassium supplement blocked the second part?

Had every patient stayed in follow-up, the high dose rather than no pill would change readmission
by day 90 by -18.6 per 100 patients (95% confidence interval -21.6 to -15.5), and death before any
readmission by +23.4 per 100 (20.7 to 26.2). The fluid part alone, the high dose with the
supplement, would change readmission by -13.6 per 100 (-16.8 to -10.3); the other -5.0 per 100
comes from patients who died first. That split rests on conditions no data can check: the
supplement blocks the potassium part and nothing else, and no unrecorded factor, such as frailty,
raises both readmission and death.

The data are simulated, so we know the true changes (-17.7, +23.1 and -13.1 per 100), and each
interval contains its value.

## The question

The clinic decides the dose when the patient goes home; discharge is time zero, when follow-up
starts. Follow-up ends at the first readmission or death. They are competing events: a patient
who dies can no longer be readmitted. The quantity to estimate is the share readmitted, and the
share who die before any readmission, by day 90 under three plans: every patient on the high
dose, no patient on the pill, and every patient on the high dose with a supplement that keeps
potassium at its level without the pill.

```text
  older, prior_admissions ──▶ dose ──▶ fluid part ──────▶ readmission
           │                       └──▶ potassium part ──▶ death
           └──────────────────────────────────────────▶ readmission, death, leaving follow-up
  assumed absent: fluid part ──▶ death; potassium part ──▶ readmission;
                  an unrecorded cause of the dose and an event, or of both events
```

Doctors gave higher doses to patients with more admissions in the past year and lower doses to
patients aged 75 or older, and both characteristics also change the chance of each event. They
are confounders, variables that drive both the dose and the outcome. The clinic checks on
patients on days 30, 60 and 90, and a patient can leave follow-up after a check, which is
censoring: an event may happen later but the data hide it.

No patient got the supplement. The arrows marked absent are what let us compute its plan anyway,
from the readmission chance of high-dose patients still alive and the death chance of patients
on no pill. Stensrud and colleagues (2022) call this a separable effect.

## The data

Every fit sets its seeds, so a rerun prints the same numbers.

In [1]:
from dataclasses import replace

import pandas as pd
from sklearn.linear_model import LogisticRegression

import cleverly

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.2.1


`make_heart_failure` draws 8,000 simulated patients and also returns the true answers, which
real data never give you.

In [2]:
from cleverly.datasets import make_heart_failure

# With your own data, load one row per patient, such as pd.read_csv(...), with the baseline
# characteristics, the dose, the day of the first event or of the last follow-up check, and an
# event code: 1 for a readmission, 2 for a death, 0 for neither by that day.
# Real data have no `truth`: delete the lines marked "simulation only" here and below.
frame, truth = make_heart_failure(n=8_000, seed=11)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().to_string())
print()
print("patients by dose:", frame["dose"].value_counts().sort_index().to_dict())
labels = {0: "neither", 1: "readmission", 2: "death"}
print("first event by day 90:", frame["event"].map(labels).value_counts().to_dict())
left = frame[frame["event"].eq(0) & frame["day"].lt(90)]
print("left follow-up after the check on day:", left["day"].value_counts().sort_index().to_dict())
print()

# simulation only, to the end of the cell: the true risk of each event by day 90. In `truth`,
# readmission_dose is the dose whose fluid part every patient gets, and death_dose the dose
# whose potassium part they get.
by_parts = truth[truth["day"].eq(90)].set_index(["readmission_dose", "death_dose"])
known = pd.DataFrame(
    {
        "high dose": by_parts.loc[(2, 2)],
        "no pill": by_parts.loc[(0, 0)],
        "high dose with supplement": by_parts.loc[(2, 0)],
    }
).T[["readmission", "death"]]
print("true risk by day 90, the share of patients with each event first:")
print(known.round(4).to_string())

rows and columns: (8000, 5)
['older', 'prior_admissions', 'dose', 'day', 'event']
   older  prior_admissions  dose   day  event
0    0.0               3.0   2.0  70.0      2
1    0.0               2.0   1.0  90.0      0
2    1.0               3.0   2.0  11.0      1
3    0.0               3.0   2.0  52.0      1
4    0.0               1.0   0.0  20.0      2

patients by dose: {0.0: 2096, 1.0: 3805, 2.0: 2099}
first event by day 90: {'neither': 4204, 'readmission': 2053, 'death': 1743}
left follow-up after the check on day: {30.0: 518, 60.0: 350}

true risk by day 90, the share of patients with each event first:
                           readmission   death
high dose                       0.1866  0.3610
no pill                         0.3636  0.1300
high dose with supplement       0.2327  0.1409


Each row is one patient. When `event` is 1 (readmitted) or 2 (died), `day` is the day it
happened, such as day 70 for the death in row 0; when `event` is 0, `day` is the last check that
found the patient with neither. The supplement row of the true risks describes a world the data
never show.

## Why a simple comparison misleads

In [3]:
print("baseline means by dose:")
print(frame.groupby("dose")[["older", "prior_admissions"]].mean().round(3).to_string())
print()

# The simple comparison: the share with each recorded first event by day 90, high dose minus no
# pill.
high_or_none = frame[frame["dose"].isin([0.0, 2.0])]
shares = pd.crosstab(high_or_none["dose"], high_or_none["event"], normalize="index")
print("share with each recorded first event, high dose minus no pill:")
for event_code, cause in ((1, "readmission"), (2, "death")):
    simple = shares.loc[2.0, event_code] - shares.loc[0.0, event_code]
    line = f"  {cause:12s} simple {simple:+.4f}"
    true = known.loc["high dose", cause] - known.loc["no pill", cause]  # simulation only
    line += f", true {true:+.4f}"  # simulation only
    print(line)

baseline means by dose:
      older  prior_admissions
dose                         
0.0   0.492             1.111
1.0   0.394             1.502
2.0   0.292             1.931

share with each recorded first event, high dose minus no pill:
  readmission  simple -0.1358, true -0.1770
  death        simple +0.2099, true +0.2310


High-dose patients had more prior admissions and were less often 75 or older. Prior admissions
raise both events, so the high dose went to patients more likely to be readmitted, and the simple
readmission difference, -0.1358, falls short of the true -0.1770. The simple shares also count a
patient who left follow-up as having neither event. And no simple comparison can estimate the
supplement plan, which no patient received.

## Set up the analysis

First, describe the study (`StudyProtocol`): who is in it, when follow-up starts, what each plan
means, and how death and leaving follow-up count. The description does not change the estimate;
it records what the estimate means, and each result keeps a copy. Passing it is optional.

In [4]:
from cleverly.datasets import heart_failure_protocol

# For your own study, build cleverly.StudyProtocol(...) directly, with the fields printed below.
protocol = heart_failure_protocol()
print("\n".join(protocol.summary_lines()))

Population: Adults sent home from the clinic's hospital after an admission for heart failure
Eligibility: Age 18 years or older; Discharged home alive after a heart-failure admission
Follow-up starts: Discharge, after baseline measurement and the discharge prescription
Strategies: Prescribe the water pill at the high dose; Prescribe no water pill; Prescribe the high dose with a potassium supplement that offsets its potassium loss
Versions: The high dose, taken as prescribed or not; Usual heart-failure care without the water pill; The high dose's fluid-clearing action, with blood potassium kept at its level without the pill
Outcome: First unplanned readmission, and death before any readmission
Horizon: 90 days after discharge, with checks on days 30, 60 and 90
Intercurrent events: Treat death before readmission as a competing event: report the risk of each event first, with neither prevented; Treat leaving follow-up after a check as censoring, so the target is the risk had every patient

Each risk is the share of patients with that event first, with neither event prevented. The
third version line says what the supplement plan must mean in practice.

Next, name the columns for the day, the event code and the dose (`TimeToEvent`), the event each
code stands for (`causes=`), and the check days (`grid=`). Each event day falls in the month
that ends at the next check. `cleverly` calls a plan a regimen. `Separable` declares the
supplement plan: `arms=` gives the dose whose part acts on each event, and `first=` counts a
death before a readmission in the same month (see "Go deeper").

In [5]:
from cleverly import CausalStudy, RegimeMean, TimeToEvent
from cleverly.longitudinal import Separable

# For your own data, change the column names, the check days and the event labels.
design = TimeToEvent(
    time="day",
    event="event",
    treatment="dose",
    baseline=("older", "prior_admissions"),
    grid=(30, 60, 90),
    causes={1: "readmission", 2: "death"},
)
plans = {
    "high dose": 2,
    "no pill": 0,
    # The fluid part at the high dose and the potassium part at no pill. A death counts before a
    # readmission in the same month.
    "high dose with supplement": Separable(arms={"readmission": 2, "death": 0}, first="death"),
}
study = CausalStudy(frame, design=design, protocol=protocol)
risks = study.identify(RegimeMean(plans, horizons=(90,)))
print(risks.summary())

                         Causal question (ey_regime)
Question:   mean outcome under each     Rows:                            8,000
            declared regime             Answered by:     adjusting for history
Outcome:    event=1@30, event=1@60,
            event=1@90, event=2@30,
            event=2@60, event=2@90
            (event by each time)
Treatment:  dose
Baseline:   older, prior_admissions
What each treatment time adjusts for
treatment   adjusted for
------------------------------------------------------------------------------
dose        older, prior_admissions
Conditions the answer rests on
condition                            checkable   what it says
------------------------------------------------------------------------------
well-defined treatment               no          Each unit's observed outcomes
                                                 are the ones it would have
                                                 had under the treatments it
               

`cleverly` turns each row into an indicator of each event by each check (`event=1@30` is a
readmission by day 30) and adjusts the dose for both characteristics. Of the printed conditions,
the two that carry the most weight here are that the characteristics hold every common cause of
the dose, of leaving follow-up and of either event, and that every kind of patient had a real
chance of each compared dose and of staying in follow-up (overlap). The supplement plan adds
three:

- the two parts could be given apart, as the pill with a supplement would do;
- the fluid part changes death only by changing readmission, and the potassium part changes
  readmission only by changing death;
- no unrecorded factor, such as frailty, raises the chance of both events.

The data cannot check these three or the first three printed ones; only a trial that gave the
two parts apart could test the supplement's. The weight check below partly checks overlap.

## Estimate the risk of each event under each plan

The method, targeted maximum likelihood estimation (TMLE), models the chance of each event in
each month among patients still at risk, called a hazard. A correction step then adjusts those
models, using the treatment model (who got which dose) and the censoring model (who stays in
follow-up). The supplement plan combines the death hazard on no pill with the readmission hazard
on the high dose.

- `outcome_learner` predicts the two hazards; `survival_model="hazard"` asks for hazards, which
  the supplement plan needs.
- `treatment_learner` and `censoring_learner` predict the dose and staying in follow-up.
- `Targeting(algorithm="one_step")` runs the correction step in many small steps, the form a
  supplement plan requires.
- `CrossFitting(enabled=False)` fits every model on all rows, which suits simple models that
  cannot memorize their training rows.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, Targeting, TMLEMethod


# For your own data, swap in your learners. With flexible learners, leave out
# cross_fitting=... to keep the default, which predicts each row out of fold.
def logistic():
    return LogisticRegression(max_iter=1000, random_state=5)


method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=logistic(),
        treatment_learner=logistic(),
        censoring_learner=logistic(),
        survival_model="hazard",
    ),
    targeting=Targeting(algorithm="one_step"),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=5, n_jobs=1),
)
result = risks.estimate(method=method)
print(result.summary())

                        Longitudinal estimate (LTMLE)
Question:      the event risk under     Rows:                            8,000
               each plan; each row is   Time points:         3 (at 30, 60, 90)
               defined below            Out-of-fold predictions:            no
Outcome:       readmission at           Outcome model:     logistic regression
               event=1@30, event=1@60,                               (hazards)
               event=1@90; death at     Treatment model:   logistic regression
               event=2@30, event=2@60,  Censoring model:   logistic regression
               event=2@90 (competing    Probability limits:          [0.01, 1]
               causes)
Treatment:     dose, given at baseline
Adjusted for:  older, prior_admissions
Censoring:     day>=30, day>=60,
               day>=90; each plan
               keeps every unit
               observed
Method:        longitudinal targeted
               maximum likelihood
               (LTM

`cif_regimen[high dose, readmission @ t=3]` is the share readmitted by day 90 under the high
dose: 0.1935, against 0.3792 with no pill. The `t` counts checks, so `t=3` is day 90, and the
p-values test only whether a risk is zero. In the plan table, the supplement plan reads 2,150
patients in the last month, the high-dose and no-pill patients together, since it borrows one
hazard from each group. The bands cover all six risks at once, and `cleverly` marks them
experimental.

## Split the drop in readmission into its two sources

The common mistake is to read the whole medicine's drop in readmission as what the fluid part
does. `result.contrast(...)` compares two plans' risks, allowing for both coming from the same
patients.

In [7]:
def risk(plan, cause):
    return f"cif_regimen[{plan}, {cause} @ t=3]"


# Each difference: the first plan's risk minus the second plan's.
pairs = {
    "whole medicine": ("high dose", "no pill"),
    "fluid part": ("high dose with supplement", "no pill"),
    "potassium part": ("high dose", "high dose with supplement"),
}
effects = {}
rows = []
for cause in ("readmission", "death"):
    for part, (plan, base) in pairs.items():
        # psi holds the two named risks in order; .psi is an estimate's value and .ci its interval.
        effects[cause, part] = result.contrast(
            lambda psi: psi[0] - psi[1], [risk(plan, cause), risk(base, cause)], name=part
        )
        low, high = effects[cause, part].ci
        rows.append(
            {
                "event": cause,
                "part": part,
                "plans compared": f"{plan} - {base}",
                "change per 100": 100 * effects[cause, part].psi,
                "ci_low": 100 * low,
                "ci_high": 100 * high,
                "true": 100 * (known.loc[plan, cause] - known.loc[base, cause]),  # simulation only
            }
        )
print("change in the risk by day 90, per 100 patients:")
print(pd.DataFrame(rows).round(1).to_string(index=False))
print("label of every estimate:", sorted({result[name].support.label for name in result}))

change in the risk by day 90, per 100 patients:
      event           part                        plans compared  change per 100  ci_low  ci_high  true
readmission whole medicine                   high dose - no pill           -18.6   -21.6    -15.5 -17.7
readmission     fluid part   high dose with supplement - no pill           -13.6   -16.8    -10.3 -13.1
readmission potassium part high dose - high dose with supplement            -5.0    -5.8     -4.1  -4.6
      death whole medicine                   high dose - no pill            23.4    20.7     26.2  23.1
      death     fluid part   high dose with supplement - no pill             1.0     0.7      1.3   1.1
      death potassium part high dose - high dose with supplement            22.5    19.7     25.2  22.0
label of every estimate: ['supported']


The whole medicine changes readmission by -18.6 per 100 patients. With the potassium part
blocked, the fluid part alone changes it by -13.6 (95% CI -16.8 to -10.3). The other -5.0 per 100
(-5.8 to -4.1) is the potassium part's share: high-dose patients who died before they could be
readmitted. Reading -18.6 as the pill's benefit on readmission would count those deaths as a
benefit.

For death, the potassium part carries +22.5 of the +23.4 per 100. The fluid part adds +1.0
because patients who avoid readmission stay at risk of dying first. Every estimate reads
`supported`: a simulation study checks this kind of fit.

## Check the weights behind the estimate

The correction step weights each patient who followed a plan by one over their predicted chance
of following it and staying in follow-up. The overlap check reads these weights and the
effective sample size, the number of equally weighted patients they are worth.

In [8]:
assessment = result.assess()
ledger = assessment.to_frame()
# Keep the checks that ran: `passed` has a pass mark, `completed` is a report to read.
ran = ledger[ledger["status"].isin(["passed", "completed"])]
print(ran[["check", "status"]].to_string(index=False))
print("needs attention:", tuple(item.name for item in assessment.attention) or "none")
print()
support = assessment.report("support").to_frame()
# The weights are the same for both events, so the readmission rows show them all. Time 1 to 3
# are the months ending on days 30, 60 and 90.
weights = support[support["cause"].eq("readmission")]
columns = ["regimen", "time", "n_followed", "max_weight", "effective_n", "share_truncated"]
print("weights by plan and month:")
print(weights[columns].round(1).to_string(index=False))
ratio = (support["effective_n"] / support["n_followed"]).min()
print(f"smallest effective sample, as a share of its group: {ratio:.1%}")

          check    status
score_equations    passed
        support completed
nuisance_models completed
needs attention: none

weights by plan and month:
                  regimen  time  n_followed  max_weight  effective_n  share_truncated
                high dose     1        2099        10.4       1700.0              0.0
                high dose     2        1418        11.8       1149.2              0.0
                high dose     3         983        12.7        805.6              0.0
                  no pill     1        2096         8.7       1762.5              0.0
                  no pill     2        1558         9.3       1312.6              0.0
                  no pill     3        1167        10.1        974.4              0.0
high dose with supplement     1        4195        10.4       3460.8              0.0
high dose with supplement     2        2976        12.9       2430.1              0.0
high dose with supplement     3        2150        15.2       1727.4    

Nothing needs attention, and the correction step reached its target (`score_equations passed`).
The largest weight is 15.2, in the supplement plan's last month, every group is worth at least
80.3% of its size, and no patient's predicted chance fell to the 0.01 floor (`share_truncated`
is 0). Warning signs would be a few huge weights, patients at the floor, or an effective sample
far below the group size. A clean check reads fitted models, so it does not prove overlap or
that they are right.

## Try to break the result

These checks change the data in a known way and refit the fluid part's effect on readmission:

- `placebo` swaps in random doses that mimic the real ones, so the difference should be near
  zero;
- `random_common_cause` adds a pure-noise column, so the difference should barely move;
- `subset` refits on random 70% samples, so the refits should spread about as much as the
  standard error predicts.

None can detect a missing confounder or a broken condition of the supplement plan.

In [9]:
import warnings

from cleverly import RegimeContrast
from cleverly.exceptions import ConvergenceWarning, SparseInfluenceWarning

# refute needs the difference as a named estimate, so this fit asks for differences directly.
contrast_fit = study.identify(RegimeContrast(plans, reference="no pill", horizons=(90,))).estimate(
    method=method
)
fluid_name = "ate_regimen[high dose with supplement vs no pill, readmission @ t=3]"
print(f"fluid part, as a named estimate: {contrast_fit[fluid_name].psi:.4f}")
with warnings.catch_warnings():
    # The checks warn that the supplement plan's small change in death rests on few rows, and
    # two placebo refits, on shuffled doses, stop the correction step slightly short of its
    # target. The readings use only the readmission difference and the refits' estimates.
    warnings.simplefilter("ignore", SparseInfluenceWarning)
    warnings.simplefilter("ignore", ConvergenceWarning)
    # Each check refits 20 times, the default.
    refutation = contrast_fit.diagnostics.refute(
        estimand=fluid_name, tests=("placebo", "subset", "random_common_cause"), random_state=61
    )
columns = [
    "test",
    "original",
    "refuted_mean",
    "refuted_sd",
    "statistic_name",
    "statistic",
    "verdict",
]
print(refutation.to_frame()[columns].round(4).to_string(index=False))

fluid part, as a named estimate: -0.1359


               test  original  refuted_mean  refuted_sd                  statistic_name  statistic     verdict
            placebo   -0.1359       -0.0020      0.0140                               t    -0.6326 descriptive
             subset   -0.1359       -0.1308      0.0099               scatter ratio rho     0.9109 descriptive
random_common_cause   -0.1359       -0.1358      0.0002 paired shift in standard errors     0.0034 descriptive


All three behave as expected over 20 refits each. The random doses give a mean of -0.0020, the
noise column barely moves the estimate, and the subset refits have a scatter ratio of 0.91,
where 1 is ideal and 20 refits easily give 0.7 to 1.3. The verdict reads `descriptive`, since
no simulation study backs a pass or fail for this check on this kind of fit, so you judge the
numbers. The checks that swap in an outcome with a known effect do not run on a supplement plan.

## What if a confounder is missing?

The E-value measures how strongly a missing confounder would need to be linked to both the dose
and readmission, beyond the two recorded characteristics, to explain the effect away. It reads
the risk ratio of two plans that each set the whole dose, so it covers the whole medicine, not
the fluid part.

In [10]:
# The E-value reads two plans that each set the whole dose: here the high dose against no pill.
bound = result.sensitivity.evalue(
    (risk("high dose", "readmission"), risk("no pill", "readmission"))
)
print(bound.summary())

                                   E-value
Question:  how strong a confounder      Estimate:   rr_regimen[high dose vs no
           would explain away                         pill, readmission @ t=3]
           rr_regimen[high dose vs no   Scale:                      risk ratio
           pill, readmission @ t=3]
                             risk ratio    E-value
------------------------------------------------------------------------------
estimate                          0.510       3.33
95% limit nearest the null        0.573       2.88
An unmeasured confounder would need a risk ratio of 3.33 with both the
treatment and the outcome, beyond the measured covariates, to explain away the
estimate; 2.88 to move the interval to the null.
Warnings:
[1] No validation study licenses the E-value of the limit for this kind of
    fit, so read it as descriptive.
Notes:
[2] The E-value is the strength at which the confounder's associations with
    the treatment and with the outcome, taken as 

The high-dose readmission risk is 0.510 times the no-pill risk. A missing confounder would need
a risk ratio of 3.33 with both the dose and readmission to explain that away, and 2.88 to move
the interval's end nearest no effect onto it; whether one that strong is plausible is a judgment
about the clinic. The bound assumes the factor does not change who leaves follow-up, and warning
[1] says to read 2.88 as a description.

`cleverly` has no such bound for the supplement plan. The check that adds a made-up confounder
refuses this fit, because that confounder would raise both events and break the condition that
no factor drives both. Instead, the code leaves out each recorded characteristic in turn.

In [11]:
# For your own data, leave out each of your strongest recorded confounders in turn.
moves = []
for dropped in design.baseline:
    reduced = replace(design, baseline=tuple(n for n in design.baseline if n != dropped))
    refit = (
        CausalStudy(frame, design=reduced, protocol=protocol)
        .identify(RegimeMean(plans, horizons=(90,)))
        .estimate(method=method)
    )
    for part in ("whole medicine", "fluid part"):
        plan, base = pairs[part]
        full = effects["readmission", part]
        moved = refit.contrast(
            lambda psi: psi[0] - psi[1],
            [risk(plan, "readmission"), risk(base, "readmission")],
            name=part,
        )
        moves.append(
            {
                "left out": dropped,
                "readmission difference": part,
                "with both": full.psi,
                "left out one": moved.psi,
                "move in SEs": (moved.psi - full.psi) / full.std_error,
            }
        )
print(pd.DataFrame(moves).round(4).to_string(index=False))

        left out readmission difference  with both  left out one  move in SEs
           older         whole medicine    -0.1856       -0.1832       0.1544
           older             fluid part    -0.1359       -0.1440      -0.4896
prior_admissions         whole medicine    -0.1856       -0.1373       3.1417
prior_admissions             fluid part    -0.1359       -0.0814       3.2761


Leaving out `prior_admissions` moves the fluid part's effect from -0.1359 to -0.0814, or 3.3
standard errors, and the whole medicine's by 3.1, so the adjustment rests mostly on prior
admissions; a missing confounder as strong could move the estimates as far. Leaving out `older`
moves each by less than half a standard error. A missing factor can be stronger than either, or
push the other way.

## What we can and can't conclude

For the clinic: the high dose, compared with no pill, changes readmission by day 90 by -18.6 per
100 patients (95% interval -21.6 to -15.5) and death before any readmission by +23.4 per 100,
had every patient stayed in follow-up. Of the drop in readmission, -13.6 per 100 (-16.8 to
-10.3) is the fluid part's own effect, which a supplement that blocks the potassium part would
keep. These hold for adults sent home after a heart-failure admission, under the conditions in
"Set up the analysis" and the treatment and censoring models, and lean on prior admissions.

The supplement plan describes a treatment no patient got, so its answer is only as good as the
claim that the supplement would block the potassium part and nothing else; that needs evidence
from outside these data, such as a trial. One simulated sample does not show how the method
performs in general.

## Go deeper

### How we know the method works

Simulation studies check each method over many repeated samples.
[Separable effects](../technical-reference/method-evidence/separable-effects.md) checks this
fit: a baseline treatment with three levels, three checks, two events and leaving follow-up,
with plans that set the whole treatment and plans that set its two parts apart, at 16,000
patients. It uses another simulation, with one yes-or-no characteristic, and models that fit the
mean of each group of patients with the same characteristics. This page's logistic models, two
characteristics and 8,000 patients rest on that study and on theory. No other software that fits
this estimate was found to compare against. The
[technical entry](../technical-reference/longitudinal-tmle.md#validation-issues-special-to-this-method)
lists the method's evidence and limits; its section "Separable effects" gives the exact
definition and conditions.

### Which event counts first within a month

The data place events by month, so a death and a readmission in the same month need an order,
and when the two parts sit at different doses, the order changes the question. `first="death"`
lets the potassium part's deaths come before the fluid part's readmissions in each month. The
cell refits the supplement plan with a readmission first.

In [12]:
from cleverly.datasets import heart_failure_truth

# The same question with a readmission counted before a death in the same month.
readmission_first = {
    **plans,
    "high dose with supplement": Separable(
        arms={"readmission": 2, "death": 0}, first="readmission"
    ),
}
other_order = study.identify(RegimeMean(readmission_first, horizons=(90,))).estimate(method=method)
fluid = pairs["fluid part"]
names = [risk(plan, "readmission") for plan in fluid]
orders = {"death first": result, "readmission first": other_order}
print("fluid part, change in readmission by day 90 per 100 patients:")
for order, fit in orders.items():
    estimate = fit.contrast(lambda psi: psi[0] - psi[1], names, name=order)
    low, high = estimate.ci
    # simulation only: the true value of each order.
    first = order.split()[0]
    fluid_only, none = (
        heart_failure_truth(*doses, first=first).set_index("day").loc[90, "readmission"]
        for doses in ((2, 0), (0, 0))
    )
    true = fluid_only - none
    print(
        f"  {order:18s} {100 * estimate.psi:+5.1f}  95% CI {100 * low:+5.1f} to {100 * high:+5.1f}"
        f"  true {100 * true:+5.1f}"
    )

fluid part, change in readmission by day 90 per 100 patients:
  death first        -13.6  95% CI -16.8 to -10.3  true -13.1
  readmission first  -16.5  95% CI -19.7 to -13.4  true -15.8


The two orders give -13.6 and -16.5 per 100, each near its own true value, -13.1 and -15.8.
Neither is wrong: they answer two questions, and the data cannot say which one the clinic means,
so `cleverly` requires `first=`. The order can matter on a coarse grid like these monthly checks,
and matters little when checks are close together. Pick the order before you look at the
results, and report it with the answer.

## Where to go next

[Readmission when death can come first](longitudinal-survival.ipynb) asks a third question of the
same kind of data: readmission had death been prevented, where this page leaves death as it is
and splits the treatment. [Readmission by day 30, 60 and 90](discharge-to-readmission.ipynb)
follows one treatment and one event. The
[longitudinal TMLE reference](../technical-reference/longitudinal-tmle.md) gives the exact
definition under "Separable effects", and
[scope and refusals](../technical-reference/scope-and-refusals.md) lists what a separable fit
refuses. The [examples index](index.md) lists every example.